# BanglaHalluEval Code-Mix Dataset (Colab)
This notebook converts Bengali QA text to Latin-script Bengali code-mix.

In [1]:
# 1) Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 2) Upload or select QA CSV
from google.colab import files
from pathlib import Path

USE_UPLOAD = True  # set False to use a Drive path
DRIVE_CSV_PATH = "/content/drive/MyDrive/BanglaHalluEval/banglahallueval_qa_1000.csv"

if USE_UPLOAD:
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    input_csv_path = next(iter(uploaded.keys()))
else:
    input_csv_path = DRIVE_CSV_PATH

print("Input CSV:", input_csv_path)

Saving pilot_20_random.csv to pilot_20_random (1).csv
Input CSV: pilot_20_random (1).csv


In [3]:
# 3) Load CSV and initialize output state
import pandas as pd
import os
import csv

OUTPUT_DIR = "/content/drive/MyDrive/BanglaHalluEval/codemix_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_CSV = os.path.join(OUTPUT_DIR, "qa_1000_codemix.csv")
CHECKPOINT = os.path.join(OUTPUT_DIR, "qa_1000_codemix_checkpoint.txt")


df = pd.read_csv(input_csv_path)
required_cols = {"id", "context", "question", "correct_answer"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

print("Rows:", len(df))

processed_ids = set()
if os.path.exists(OUTPUT_CSV):
    try:
        existing = pd.read_csv(OUTPUT_CSV)
        if "id" in existing.columns:
            processed_ids = set(existing["id"].astype(str).tolist())
        print("Found existing output rows:", len(processed_ids))
    except Exception as exc:
        print("Could not read existing output, starting fresh.", exc)

last_index = -1
if os.path.exists(CHECKPOINT):
    try:
        with open(CHECKPOINT, "r", encoding="utf-8") as f:
            last_index = int(f.read().strip())
        print("Checkpoint index:", last_index)
    except Exception as exc:
        print("Could not read checkpoint.", exc)

Rows: 20


In [4]:
# 4) Define compact code-mix prompt
CODEMIX_PROMPT = (
    "Convert Bengali or English QA text into natural Banglish (Bengali-English code-mix) "
    "written entirely in Latin script — the way people write on social media or in casual chats.\n\n"

    "Rules:\n"
    "- Romanize Bengali words naturally; use English words where code-mix speakers naturally would.\n"
    "- Do NOT translate everything to English. Preserve Bengali phrasing where it feels natural.\n"
    "- Preserve meaning, tone, names, numbers, dates, brands, and technical terms exactly.\n"
    "- Keep output in Latin script only — no Bengali characters anywhere.\n"
    "- Output only the converted text. No labels, no explanations, no Bengali.\n\n"

    "Spelling variation (apply to ~15% of eligible words, not every word):\n"
    "- Use realistic Banglish alternates: bhai/vai, bhalo/valo, ache/ase/ace, "
    "ki/kee, onek/anek, ektu/aktu, shob/sob, nai/nei, hoise/hoiche, gesi/geci.\n\n"

    "Abbreviations (apply sparingly, ~10% of eligible words):\n"
    "- Common ones: phn (phone), msg (message), prob/prblm (problem), "
    "pls/plz (please), ok (okay), tmrw (tomorrow), info (information), "
    "ppt (presentation), pic (picture), sry (sorry), bcz/coz (because).\n\n"

    "Do not: over-abbreviate, use excessive expressive spellings (nooo, ufff), "
    "change factual meaning, or make output hard to read."
)

In [5]:
# 5) Conversion function and checkpointing
!pip -q install openai

import re
import time
from openai import OpenAI
from getpass import getpass

MODEL_NAME = "gpt-5.4"

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    api_key = getpass("Enter OPENAI_API_KEY: ")

client = OpenAI(api_key=api_key)

latin_only_re = re.compile(r"^[\x00-\x7F]*$")
bengali_re = re.compile(r"[\u0980-\u09FF]")


def _is_valid_latin(text: str) -> bool:
    return bool(text) and latin_only_re.match(text) and not bengali_re.search(text)


def to_codemix(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""

    base_prompt = f"{CODEMIX_PROMPT}\n\nInput: {text}\nOutput:"
    strict_prompt = (
        f"{CODEMIX_PROMPT} STRICT: Output must be Latin-only. "
        f"If any Bengali characters appear, redo.\n\nInput: {text}\nOutput:"
    )
    repair_prompt = (
        "Convert the following text to Latin-only Banglish. "
        "Remove all Bengali characters. Output only the converted text.\n\nText: "
        + text
        + "\nOutput:"
    )

    prompts = [base_prompt, strict_prompt, strict_prompt, repair_prompt, strict_prompt]
    retry_count = 5
    if len(prompts) < retry_count:
        prompts.extend([strict_prompt] * (retry_count - len(prompts)))
    last_non_empty = ""
    for attempt, prompt in enumerate(prompts[:retry_count], start=1):
        try:
            resp = client.responses.create(
                model=MODEL_NAME,
                input=[{"role": "user", "content": prompt}],
                max_output_tokens=200,
                temperature=0.2,
            )
            out = (resp.output_text or "").strip()
            if out:
                last_non_empty = out
            if _is_valid_latin(out):
                return out
        except Exception as exc:
            if attempt == retry_count:
                print("Error:", exc)
        time.sleep(0.5 + (attempt - 1) * 0.5)

    # Final fallback: return last non-empty output even if Bengali remains
    if last_non_empty:
        return last_non_empty

    return "NA"


def append_row(row_dict: dict, output_csv: str):
    file_exists = os.path.exists(output_csv)
    with open(output_csv, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row_dict.keys()))
        if not file_exists or os.path.getsize(output_csv) == 0:
            writer.writeheader()
        writer.writerow(row_dict)
        f.flush()


def update_checkpoint(index: int, checkpoint_path: str):
    with open(checkpoint_path, "w", encoding="utf-8") as f:
        f.write(str(index))

Enter OPENAI_API_KEY: ··········


In [6]:
# 6) Process dataset with resume
BATCH_SIZE = 50
SLEEP_BETWEEN = 0.2

start_idx = max(last_index + 1, 0)

for i in range(start_idx, len(df)):
    row = df.iloc[i]
    row_id = str(row["id"])
    if row_id in processed_ids:
        continue

    codemix_context = to_codemix(row["context"])
    codemix_question = to_codemix(row["question"])
    codemix_answer = to_codemix(row["correct_answer"])

    out_row = {
        "id": row_id,
        "context": row["context"],
        "question": row["question"],
        "correct_answer": row["correct_answer"],
        "codemix_context": codemix_context,
        "codemix_question": codemix_question,
        "codemix_answer": codemix_answer,
    }

    append_row(out_row, OUTPUT_CSV)
    update_checkpoint(i, CHECKPOINT)

    if (i + 1) % BATCH_SIZE == 0:
        print("Processed:", i + 1)
        time.sleep(SLEEP_BETWEEN)

print("Done. Output:", OUTPUT_CSV)

Done. Output: /content/drive/MyDrive/BanglaHalluEval/codemix_output/qa_1000_codemix.csv
